# Agentic AI Sycophancy: GitHub Data Collection

This notebook implements the final Stage-1 harvest of a two-stage design for a taxonomy-building study of sycophancy in agentic AI systems, and constructs the sampling frame for Stage 2 (stratified screening and multi-label coding, handled separately).

The notebook:
1. verifies and canonicalises 80 target repositories (six functional categories) against the GitHub API,
2. searches public GitHub issues whose creation date falls between 1 January 2021 and 30 June 2026, matching issue titles, bodies, AND comments (`in:title,body,comments`),
3. uses bundled OR-queries per dimension and layer (core / exploratory) rather than one query per term, splitting date windows recursively at GitHub's 1,000-result cap and logging every executed query,
4. removes duplicate issues (recording all matching bundle dimensions and levels),
5. retrieves all public comments for every unique candidate issue with full pagination and checkpointing, labelling zero-comment issues,
6. constructs one structured thread text per issue with author, role, and timestamp markers, then performs local term attribution: every lexicon term is matched against the full thread text, giving term-, dimension-, and level-resolved attribution per thread, and
7. builds a seeded Stage-2 sampling frame with repository caps (max 40 per repository), dimension floors (min 30 where available), an advisory repository-share check (20%), and a reserve pool for sensitivity analysis.

The resulting dataset is a keyword-enriched candidate corpus for taxonomy development and empirical mapping. It is not a prevalence sample.

API keys are not included. Add `GITHUB_TOKEN` through Google Colab Secrets before running the notebook.


In [ ]:
!pip install -q requests pandas tqdm openpyxl

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    print("No GitHub token found. Public API access may still work, but rate limits will be much lower.")
else:
    print("GitHub token loaded.")

In [ ]:
HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"


In [ ]:
# Final corpus: 80 repositories across six functional categories, expanded to
# strengthen browser, research, memory, and multi-agent coverage after two pilot
# runs. Repo paths are canonicalised in the next cell; unresolved paths are
# reported and skipped.

REPO_CATEGORIES = {
    # 1. Agent SDKs and orchestration frameworks (13)
    "microsoft/agent-framework": "Agent SDKs and orchestration",
    "microsoft/semantic-kernel": "Agent SDKs and orchestration",
    "langchain-ai/langgraph": "Agent SDKs and orchestration",
    "openai/openai-agents-python": "Agent SDKs and orchestration",
    "google/adk-python": "Agent SDKs and orchestration",
    "langchain-ai/langchain": "Agent SDKs and orchestration",
    "Significant-Gravitas/AutoGPT": "Agent SDKs and orchestration",
    "pydantic/pydantic-ai": "Agent SDKs and orchestration",
    "agno-agi/agno": "Agent SDKs and orchestration",
    "mastra-ai/mastra": "Agent SDKs and orchestration",
    "huggingface/smolagents": "Agent SDKs and orchestration",
    "deepset-ai/haystack": "Agent SDKs and orchestration",
    "FlowiseAI/Flowise": "Agent SDKs and orchestration",

    # 2. Multi-agent collaboration and coordination (14)
    "microsoft/autogen": "Multi-agent collaboration",
    "crewAIInc/crewAI": "Multi-agent collaboration",
    "camel-ai/camel": "Multi-agent collaboration",
    "FoundationAgents/MetaGPT": "Multi-agent collaboration",
    "OpenBMB/ChatDev": "Multi-agent collaboration",
    "agentscope-ai/agentscope": "Multi-agent collaboration",
    "agentscope-ai/AgentTeams": "Multi-agent collaboration",
    "TransformerOptimus/SuperAGI": "Multi-agent collaboration",
    "VoltAgent/voltagent": "Multi-agent collaboration",
    "openai/swarm": "Multi-agent collaboration",
    "langroid/langroid": "Multi-agent collaboration",
    "awslabs/agent-squad": "Multi-agent collaboration",
    "kyegomez/swarms": "Multi-agent collaboration",
    "microsoft/TinyTroupe": "Multi-agent collaboration",

    # 3. Coding and software-engineering agents (12)
    "OpenHands/OpenHands": "Coding and software-engineering agents",
    "cline/cline": "Coding and software-engineering agents",
    "openai/codex": "Coding and software-engineering agents",
    "google-gemini/gemini-cli": "Coding and software-engineering agents",
    "anthropics/claude-code": "Coding and software-engineering agents",
    "SWE-agent/SWE-agent": "Coding and software-engineering agents",
    "Aider-AI/aider": "Coding and software-engineering agents",
    "continuedev/continue": "Coding and software-engineering agents",
    "TabbyML/tabby": "Coding and software-engineering agents",
    "RooCodeInc/Roo-Code": "Coding and software-engineering agents",
    "getcursor/cursor": "Coding and software-engineering agents",
    "plandex-ai/plandex": "Coding and software-engineering agents",

    # 4. Browser, computer-use, research, and action agents (19)
    "browser-use/browser-use": "Browser, research, and action agents",
    "FoundationAgents/OpenManus": "Browser, research, and action agents",
    "assafelovic/gpt-researcher": "Browser, research, and action agents",
    "langchain-ai/open_deep_research": "Browser, research, and action agents",
    "SamuelSchmidgall/AgentLaboratory": "Browser, research, and action agents",
    "Skyvern-AI/skyvern": "Browser, research, and action agents",
    "nanobrowser/nanobrowser": "Browser, research, and action agents",
    "vercel-labs/agent-browser": "Browser, research, and action agents",
    "browser-use/web-ui": "Browser, research, and action agents",
    "run-llama/llama_index": "Browser, research, and action agents",
    "browserbase/stagehand": "Browser, research, and action agents",
    "lavague-ai/LaVague": "Browser, research, and action agents",
    "microsoft/OmniParser": "Browser, research, and action agents",
    "trycua/cua": "Browser, research, and action agents",
    "simular-ai/Agent-S": "Browser, research, and action agents",
    "stanford-oval/storm": "Browser, research, and action agents",
    "dzhng/deep-research": "Browser, research, and action agents",
    "camel-ai/owl": "Browser, research, and action agents",
    "bytedance/deer-flow": "Browser, research, and action agents",

    # 5. Memory, personalisation, and stateful agent infrastructure (11)
    "mem0ai/mem0": "Memory and personalisation infrastructure",
    "letta-ai/letta": "Memory and personalisation infrastructure",
    "getzep/graphiti": "Memory and personalisation infrastructure",
    "langchain-ai/langmem": "Memory and personalisation infrastructure",
    "langgenius/dify": "Memory and personalisation infrastructure",
    "getzep/zep": "Memory and personalisation infrastructure",
    "topoteretes/cognee": "Memory and personalisation infrastructure",
    "supermemoryai/supermemory": "Memory and personalisation infrastructure",
    "kingjulio8238/memary": "Memory and personalisation infrastructure",
    "memodb-io/memobase": "Memory and personalisation infrastructure",
    "basicmachines-co/basic-memory": "Memory and personalisation infrastructure",

    # 6. Evaluation, observability, guardrails, and control (11)
    "promptfoo/promptfoo": "Evaluation, observability, and control",
    "confident-ai/deepeval": "Evaluation, observability, and control",
    "langfuse/langfuse": "Evaluation, observability, and control",
    "Arize-ai/phoenix": "Evaluation, observability, and control",
    "AgentOps-AI/agentops": "Evaluation, observability, and control",
    "openai/evals": "Evaluation, observability, and control",
    "vibrantlabsai/ragas": "Evaluation, observability, and control",
    "guardrails-ai/guardrails": "Evaluation, observability, and control",
    "NVIDIA-NeMo/Guardrails": "Evaluation, observability, and control",
    "Giskard-AI/giskard": "Evaluation, observability, and control",
    "comet-ml/opik": "Evaluation, observability, and control",
}

REPOS = list(REPO_CATEGORIES.keys())

# Final search lexicon: eight dimensions, each with a core (high-precision) and
# an exploratory (broader discovery) layer. Terms are searched as OR-bundles per
# dimension and layer (see the search cell); after retrieval, term-level
# attribution is performed locally against the full thread text.
SYCOPHANCY_TERMS = {
    "Direct Sycophancy and Agreement": {
        "core": [
            "sycophancy", "sycophantic", "pandering", "people pleasing",
            "overly agreeable", "too agreeable", "excessive praise",
            "flattery", "you're absolutely right", "yes-man"
        ],
        "exploratory": [
            "agrees with everything", "always agrees", "excessive agreement",
            "tells the user what they want to hear", "blind agreement",
            "user is always right"
        ]
    },
    "Opinion Conformity and Belief Reinforcement": {
        "core": [
            "uncritically agrees", "accepts a false premise",
            "reinforces a misconception"
        ],
        "exploratory": [
            "mirrors the user", "echoes the user", "accepts an incorrect premise",
            "confirms the user's belief", "reinforces an incorrect belief",
            "echo chamber", "self-reinforcing agreement"
        ]
    },
    "Capitulation and Answer Flipping": {
        "core": [
            "changes answer after pushback", "changes answer when challenged",
            "reverses a correct answer", "backs down when challenged",
            "abandons the correct answer", "capitulates"
        ],
        "exploratory": [
            "apologizes and changes answer", "apologises and changes answer",
            "second-guesses a correct answer", "gives in to user pressure",
            "retreats from a justified position", "flip-flop"
        ]
    },
    "Evidence and Reasoning Conformity": {
        "core": [
            "cherry-picks evidence", "ignores contradictory evidence",
            "suppresses contrary evidence", "confirmation bias"
        ],
        "exploratory": [
            "selective evidence", "selective retrieval", "one-sided summary",
            "one-sided evidence", "biased summary", "slanted summary",
            "predetermined conclusion", "confirms the user's conclusion",
            "researches only one side"
        ]
    },
    "Agentic Action and Plan Sycophancy": {
        "core": [
            "fails to challenge", "blindly follows", "accepts a bad plan",
            "uncritical compliance", "validates a bad idea",
            "changes plan to please"
        ],
        "exploratory": [
            "does not push back", "ignores constraints",
            "ignores user-defined constraints", "executes an unsafe request",
            "follows an unreasonable instruction", "overweights user preference",
            "continues despite contradictory evidence"
        ]
    },
    "Inter-Agent Deference and Consensus": {
        "core": [
            "agent deference", "premature consensus", "consensus collapse",
            "disagreement collapse", "uncritically accepts the planner",
            "blindly follows the supervisor"
        ],
        "exploratory": [
            "agent herding", "groupthink", "false consensus", "rubber-stamps",
            "inter-agent conformity", "suppresses dissent",
            "agrees with the supervisor", "fails to challenge another agent"
        ]
    },
    "Personalisation and Memory Sycophancy": {
        "core": [
            "memory-induced sycophancy", "personalisation bias",
            "personalization bias", "uses preference as fact",
            "preference overrides evidence"
        ],
        "exploratory": [
            "stale preference", "outdated user preference",
            "remembered belief as fact", "reinforces remembered belief",
            "over-personalisation", "over-personalization",
            "refuses to update a user model", "preference overrides current instruction"
        ]
    },
    "Evaluator and Judge Conformity": {
        "core": [
            "evaluator agrees with the model", "judge favours its own answer",
            "judge favors its own answer", "critic fails to challenge"
        ],
        "exploratory": [
            "grade inflation", "inflated score", "score inflation",
            "lenient evaluation", "evaluator bias", "self-preference",
            "positive evaluation bias", "overly generous judging"
        ]
    }
}

n_terms = sum(len(tl) for d in SYCOPHANCY_TERMS.values() for tl in d.values())
n_core = sum(len(d["core"]) for d in SYCOPHANCY_TERMS.values())
print(f"{len(REPOS)} repositories, {n_terms} search terms "
      f"({n_core} core, {n_terms - n_core} exploratory) in "
      f"{sum(len(d) for d in SYCOPHANCY_TERMS.values())} dimension-level bundles.")


In [ ]:
# Verify and canonicalise repository names before searching.
# GitHub's search API can silently return zero results for renamed or moved
# repositories, so every path is resolved to its current canonical location,
# and a full resolution report is printed. The number of repositories actually
# searched is the resolved count, which may be lower than the supplied count.

import requests
import time

n_supplied = len(REPO_CATEGORIES)
VERIFIED_REPO_CATEGORIES = {}
renamed = []
skipped = []

for repo, category in REPO_CATEGORIES.items():
    response = requests.get(f"https://api.github.com/repos/{repo}", headers=HEADERS)

    if response.status_code == 200:
        canonical = response.json().get("full_name", repo)
        if canonical != repo:
            renamed.append((repo, canonical))
        VERIFIED_REPO_CATEGORIES[canonical] = category
    else:
        skipped.append((repo, response.status_code))

    time.sleep(0.5)

REPOS = list(VERIFIED_REPO_CATEGORIES.keys())
REPO_CATEGORIES = VERIFIED_REPO_CATEGORIES

N_SUPPLIED = n_supplied
N_RESOLVED = len(REPOS)
N_SKIPPED = len(skipped)

print("Repository resolution report")
print(f"  Supplied: {N_SUPPLIED}")
print(f"  Successfully resolved: {N_RESOLVED}")
print(f"  Skipped (unresolvable): {N_SKIPPED}")

if renamed:
    print("\n  Renamed/moved (canonical path will be searched):")
    for old, new in renamed:
        print(f"    {old} -> {new}")

if skipped:
    print("\n  Skipped repositories (verify these paths manually on github.com):")
    for repo, status in skipped:
        print(f"    {repo}: HTTP {status}")

print(f"\nFinal number of repositories to be searched: {N_RESOLVED}")
print("Report the RESOLVED count, not the supplied count, in the Methods.")


In [ ]:
# Reproducibility settings

from datetime import date

# Fixed collection window. Eligibility is based on ISSUE CREATION DATE:
# only issues created within this window are collected.
START_DATE = date(2021, 1, 1)
END_DATE = date(2026, 6, 30)

SEARCH_SLEEP_SECONDS = 2

# GitHub search returns at most 1,000 results per query (10 pages of 100).
# Queries exceeding the cap are split recursively into smaller date windows;
# queries approaching the cap are flagged in the search log.
PER_PAGE = 100
MAX_PAGES_PER_QUERY = 10
API_RESULT_CAP = 1000
NEAR_CAP_THRESHOLD = 900

# Bundled queries must stay under GitHub's query-length limit
MAX_QUERY_CHARS = 250

# Stage-2 sampling frame parameters.
# IMPORTANT: these parameters affect ONLY the optional selected_for_coding flag
# in the sampling frame. The primary outputs (all unique candidate issues, all
# comments, all issue threads) are always complete and are never reduced.
# MAX_PER_REPO = None disables the repository cap entirely (every thread is
# selected and the reserve pool is empty); set an integer (e.g. 40) to produce
# a balanced coding sample instead.
RANDOM_SEED = 42
MAX_PER_REPO = None
MIN_PER_DIMENSION = 30
MAX_REPO_SHARE = 0.20   # advisory: reported for the selected set, never enforced

# Output filenames
CANDIDATE_CSV = "github_sycophancy_candidate_issues.csv"
CANDIDATE_EXCEL = "github_sycophancy_candidate_issues_clean.xlsx"

COMMENTS_CSV = "github_sycophancy_issue_comments.csv"
COMMENTS_EXCEL = "github_sycophancy_issue_comments.xlsx"

THREADS_CSV = "github_sycophancy_issue_threads.csv"
THREADS_EXCEL = "github_sycophancy_issue_threads.xlsx"

SAMPLING_FRAME_CSV = "github_sycophancy_sampling_frame.csv"
SAMPLING_FRAME_EXCEL = "github_sycophancy_sampling_frame.xlsx"

SEARCH_LOG_CSV = "github_sycophancy_search_log.csv"


In [ ]:
# Optional: corpus size estimate (dry run)
#
# Set RUN_DRY_RUN = True to query only the total_count for every repo-bundle
# pair (one API request each, no pagination). This gives an exact
# pre-deduplication corpus size estimate before committing to full collection.
# STRONGLY RECOMMENDED before the full harvest: comment-text matching can make
# some bundles very large in high-volume repositories.

RUN_DRY_RUN = False


def or_clause(terms):
    return "(" + " OR ".join(f'"{t}"' for t in terms) + ")"


def pack_terms(repo, terms, window_start, window_end, max_chars=MAX_QUERY_CHARS):
    base = (f"repo:{repo} is:issue in:title,body,comments "
            f"created:{window_start.isoformat()}..{window_end.isoformat()} ")
    chunks = []
    current = []
    for term in terms:
        candidate = current + [term]
        if len(base + or_clause(candidate)) > max_chars and current:
            chunks.append(current)
            current = [term]
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

if RUN_DRY_RUN:
    import time
    import requests
    import pandas as pd
    from tqdm import tqdm

    count_rows = []

    for repo in tqdm(REPOS, desc="Repositories"):
        for dimension, levels in SYCOPHANCY_TERMS.items():
            for level, terms in levels.items():
                for chunk_index, chunk_terms in enumerate(
                        pack_terms(repo, terms, START_DATE, END_DATE)):
                    query = (
                        f"repo:{repo} is:issue in:title,body,comments "
                        f"created:{START_DATE.isoformat()}..{END_DATE.isoformat()} "
                        + or_clause(chunk_terms)
                    )
                    response = requests.get(
                        "https://api.github.com/search/issues",
                        headers=HEADERS,
                        params={"q": query, "advanced_search": "true", "per_page": 1}
                    )
                    if response.status_code == 200:
                        total = response.json().get("total_count")
                    else:
                        total = None
                        if response.status_code in [403, 429]:
                            time.sleep(60)

                    count_rows.append({
                        "repo": repo,
                        "dimension": dimension,
                        "bundle_level": level,
                        "bundle_terms": "; ".join(chunk_terms),
                        "total_count": total
                    })
                    time.sleep(SEARCH_SLEEP_SECONDS)

    counts_df = pd.DataFrame(count_rows)
    counts_df.to_csv("github_sycophancy_dry_run_counts.csv",
                     index=False, encoding="utf-8-sig")

    print(f"Total raw hits (before deduplication): {counts_df['total_count'].sum():,.0f}")
    print("\nTop 20 repo-bundle pairs by hit count:")
    display(counts_df.sort_values("total_count", ascending=False).head(20))
    print("\nHits by dimension and level:")
    display(counts_df.groupby(["dimension", "bundle_level"])["total_count"].sum())


In [ ]:
import math
import pandas as pd
from datetime import timedelta
from tqdm import tqdm


def github_get(url, params=None, max_retries=5):
    for attempt in range(max_retries):
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code == 200:
            return response.json()

        if response.status_code in [403, 429]:
            reset_time = response.headers.get("x-ratelimit-reset")
            remaining = response.headers.get("x-ratelimit-remaining")

            if remaining == "0" and reset_time:
                sleep_for = max(int(reset_time) - int(time.time()) + 5, 10)
                print(f"Rate limit reached. Sleeping for {sleep_for} seconds.")
                time.sleep(sleep_for)
            else:
                wait = 60 * (attempt + 1)
                print(f"Secondary rate limit. Sleeping for {wait} seconds.")
                time.sleep(wait)
            continue

        print(f"Request failed: {response.status_code}")
        print(response.text[:300])
        return None

    return None


SEARCH_URL = "https://api.github.com/search/issues"


def or_clause(terms):
    return "(" + " OR ".join(f'"{t}"' for t in terms) + ")"


def pack_terms(repo, terms, window_start, window_end, max_chars=MAX_QUERY_CHARS):
    """
    Greedily pack terms into OR-bundles whose full query strings stay under
    GitHub's query-length limit.
    """
    base = (f"repo:{repo} is:issue in:title,body,comments "
            f"created:{window_start.isoformat()}..{window_end.isoformat()} ")
    chunks = []
    current = []

    for term in terms:
        candidate = current + [term]
        if len(base + or_clause(candidate)) > max_chars and current:
            chunks.append(current)
            current = [term]
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def item_to_row(item, repo, dimension, level, bundle_id, bundle_terms):
    return {
        "bundle_dimension": dimension,
        "bundle_level": level,
        "bundle_id": bundle_id,
        "bundle_terms": "; ".join(bundle_terms),
        "repo": repo,
        "repo_category": REPO_CATEGORIES.get(repo, ""),
        "issue_number": item.get("number"),
        "title": item.get("title"),
        "body": item.get("body"),
        "issue_author": item.get("user", {}).get("login") if item.get("user") else None,
        "issue_author_association": item.get("author_association"),
        "state": item.get("state"),
        "created_at": item.get("created_at"),
        "updated_at": item.get("updated_at"),
        "closed_at": item.get("closed_at"),
        "comments_count": item.get("comments"),
        "labels": "; ".join([label["name"] for label in item.get("labels", [])]),
        "html_url": item.get("html_url"),
        "comments_url": item.get("comments_url")
    }


def search_bundle_window(repo, dimension, level, bundle_id, bundle_terms,
                         window_start, window_end, search_log):
    """
    Exhaustively retrieve all issues matching an OR-bundle of terms within a
    date window, matching issue titles, bodies, AND comments. Windows over
    GitHub's 1,000-result cap are split recursively. Every executed query is
    logged.
    """
    rows = []
    query = (
        f"repo:{repo} is:issue in:title,body,comments "
        f"created:{window_start.isoformat()}..{window_end.isoformat()} "
        + or_clause(bundle_terms)
    )

    # advanced_search enables boolean OR syntax on the issues search endpoint
    params = {"q": query, "advanced_search": "true", "sort": "created",
              "order": "asc", "per_page": PER_PAGE, "page": 1}
    data = github_get(SEARCH_URL, params=params)
    time.sleep(SEARCH_SLEEP_SECONDS)

    log_base = {
        "repo": repo, "dimension": dimension, "bundle_level": level,
        "bundle_id": bundle_id, "n_terms": len(bundle_terms),
        "query_chars": len(query),
        "window_start": window_start.isoformat(),
        "window_end": window_end.isoformat()
    }

    if not data or "items" not in data:
        search_log.append({**log_base, "total_count": None, "retrieved": 0,
                           "action": "request_failed", "truncated": True,
                           "near_cap": False})
        return rows

    total_count = data.get("total_count", 0)

    if total_count > API_RESULT_CAP and window_start < window_end:
        mid = window_start + (window_end - window_start) / 2
        search_log.append({**log_base, "total_count": total_count, "retrieved": 0,
                           "action": "split", "truncated": False, "near_cap": False})
        rows.extend(search_bundle_window(
            repo, dimension, level, bundle_id, bundle_terms,
            window_start, mid, search_log))
        rows.extend(search_bundle_window(
            repo, dimension, level, bundle_id, bundle_terms,
            mid + timedelta(days=1), window_end, search_log))
        return rows

    for item in data["items"]:
        rows.append(item_to_row(item, repo, dimension, level, bundle_id, bundle_terms))

    pages_needed = min(math.ceil(total_count / PER_PAGE), MAX_PAGES_PER_QUERY)

    for page in range(2, pages_needed + 1):
        params["page"] = page
        data = github_get(SEARCH_URL, params=params)
        time.sleep(SEARCH_SLEEP_SECONDS)

        if not data or "items" not in data or len(data["items"]) == 0:
            break

        for item in data["items"]:
            rows.append(item_to_row(item, repo, dimension, level, bundle_id, bundle_terms))

    search_log.append({**log_base, "total_count": total_count, "retrieved": len(rows),
                       "action": "collected",
                       "truncated": total_count > API_RESULT_CAP,
                       "near_cap": total_count >= NEAR_CAP_THRESHOLD})
    return rows


all_rows = []
search_log = []

for repo in tqdm(REPOS, desc="Repositories"):
    for dimension, levels in SYCOPHANCY_TERMS.items():
        for level, terms in levels.items():
            chunks = pack_terms(repo, terms, START_DATE, END_DATE)
            for chunk_index, chunk_terms in enumerate(chunks):
                bundle_id = f"{dimension} | {level} | {chunk_index + 1}"
                all_rows.extend(search_bundle_window(
                    repo, dimension, level, bundle_id, chunk_terms,
                    START_DATE, END_DATE, search_log))

search_log_df = pd.DataFrame(search_log)
search_log_df.to_csv(SEARCH_LOG_CSV, index=False, encoding="utf-8-sig")

print(f"Executed queries logged: {len(search_log_df)}")
print(f"Truncated or failed queries: {int(search_log_df['truncated'].sum())}")
print(f"Queries near the 1,000-result cap: {int(search_log_df['near_cap'].sum())}")

issues_df = pd.DataFrame(all_rows)

# Duplicates arise across bundles and adjacent date windows. Before removal,
# record every bundle dimension and level that retrieved each issue.
bundle_map = (
    issues_df
    .groupby(["repo", "issue_number"])
    .agg(
        all_bundle_dimensions=("bundle_dimension", lambda s: "; ".join(sorted(set(s)))),
        all_bundle_levels=("bundle_level", lambda s: "; ".join(sorted(set(s)))),
        n_bundles=("bundle_id", "nunique")
    )
    .reset_index()
)

issues_df = issues_df.drop_duplicates(subset=["repo", "issue_number"])
issues_df = issues_df.merge(bundle_map, on=["repo", "issue_number"], how="left")

issues_df.to_csv(CANDIDATE_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(issues_df)} unique candidate issues.")
issues_df.head()


In [ ]:
import re

# If issues_df is still in memory, use it.
# If not, reload from the CSV that was already saved.
try:
    issues_df
    print("Using existing issues_df in memory.")
except NameError:
    issues_df = pd.read_csv(CANDIDATE_CSV)
    print("Reloaded issues_df from CSV.")

# Remove illegal Excel characters
ILLEGAL_CHARACTERS_RE = re.compile(r"[\000-\010]|[\013-\014]|[\016-\037]")

def clean_excel_text(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

issues_df_clean = issues_df.applymap(clean_excel_text)

# Save cleaned Excel version
issues_df_clean.to_excel(CANDIDATE_EXCEL, index=False)

print(f"Saved cleaned Excel file with {len(issues_df_clean)} unique candidate issues.")


In [ ]:
from google.colab import files

files.download(CANDIDATE_EXCEL)
files.download(CANDIDATE_CSV)
files.download(SEARCH_LOG_CSV)


In [ ]:
# Load the cleaned candidate issue dataset
issues_df = pd.read_excel(CANDIDATE_EXCEL)

issues_df["comments_count"] = pd.to_numeric(
    issues_df["comments_count"], errors="coerce"
).fillna(0).astype(int)

# Zero-comment issues are labelled (not excluded), so downstream screening can
# handle report-only threads separately from discussed threads.
issues_df["zero_comments"] = (issues_df["comments_count"] == 0).astype(int)

# All unique candidate issues proceed to comment retrieval and thread construction.
issues_for_coding = issues_df.copy().reset_index(drop=True)

print("Candidate issues by repository category:")
display(issues_for_coding["repo_category"].value_counts())

print("\nCandidate issues by bundle dimension (first-retrieved):")
display(issues_for_coding["bundle_dimension"].value_counts())

print("\nCandidate issues by bundle level (any):")
for level in ["core", "exploratory"]:
    n = issues_for_coding["all_bundle_levels"].fillna("").str.contains(level).sum()
    print(f"  {level}: {n}")

print(f"\nZero-comment issues (labelled, retained): {issues_for_coding['zero_comments'].sum()}")
print(f"Prepared {len(issues_for_coding)} unique candidate issues for comment retrieval.")


In [ ]:
def get_issue_comments(comments_url):
    comments = []
    page = 1

    while True:
        params = {
            "per_page": 100,
            "page": page
        }

        data = github_get(comments_url, params=params)

        if not data or len(data) == 0:
            break

        for comment in data:
            comments.append({
                "comment_id": comment.get("id"),
                "comment_author": comment.get("user", {}).get("login") if comment.get("user") else None,
                "comment_author_association": comment.get("author_association"),
                "comment_created_at": comment.get("created_at"),
                "comment_updated_at": comment.get("updated_at"),
                "comment_body": comment.get("body"),
                "comment_url": comment.get("html_url")
            })

        page += 1
        time.sleep(1)

    return comments


# Comment retrieval is the longest stage. Progress is checkpointed to
# COMMENTS_CSV every CHECKPOINT_EVERY processed issues, and already-retrieved
# issues are skipped on rerun, so an interrupted session resumes without loss.
CHECKPOINT_EVERY = 200

import os

if os.path.exists(COMMENTS_CSV):
    existing_comments_df = pd.read_csv(COMMENTS_CSV)
    all_comments = existing_comments_df.to_dict("records")
    done_issues = set(zip(existing_comments_df["repo"],
                          existing_comments_df["issue_number"]))
    print(f"Resuming: comments for {len(done_issues)} issues already retrieved.")
else:
    all_comments = []
    done_issues = set()

processed_since_checkpoint = 0

for _, row in tqdm(issues_for_coding.iterrows(), total=len(issues_for_coding)):
    comments_url = row["comments_url"]

    # Issues with no comments have nothing to retrieve
    if pd.isna(comments_url) or row["comments_count"] == 0:
        continue

    if (row["repo"], row["issue_number"]) in done_issues:
        continue

    comments = get_issue_comments(comments_url)

    for comment in comments:
        comment.update({
            "repo": row["repo"],
            "repo_category": row["repo_category"],
            "issue_number": row["issue_number"],
            "issue_title": row["title"],
            "issue_url": row["html_url"],
            "bundle_dimension": row["bundle_dimension"],
            "bundle_level": row["bundle_level"]
        })

        all_comments.append(comment)

    processed_since_checkpoint += 1
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        pd.DataFrame(all_comments).to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")
        processed_since_checkpoint = 0

comments_df = pd.DataFrame(all_comments)

comments_df.to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(comments_df)} comments from {len(issues_for_coding)} candidate issues.")
comments_df.head()


In [ ]:
# Thread construction with local term attribution.
#
# Bundled OR-queries establish that an issue matched a bundle somewhere in its
# title, body, or comments, but not which term. After assembling the full
# thread text, each lexicon term is matched locally (case-insensitive substring,
# with typographic apostrophes normalised), giving term-, dimension-, and
# level-resolved attribution per thread.

def attribute_terms(text):
    if not isinstance(text, str):
        return [], [], []
    t = text.lower().replace("\u2019", "'")
    matched_terms = []
    matched_dims = set()
    matched_levels = set()
    for dim, levels in SYCOPHANCY_TERMS.items():
        for level, term_list in levels.items():
            for term in term_list:
                if term.lower() in t:
                    matched_terms.append(term)
                    matched_dims.add(dim)
                    matched_levels.add(level)
    return matched_terms, sorted(matched_dims), sorted(matched_levels)


thread_rows = []

for _, issue in issues_for_coding.iterrows():
    repo = issue["repo"]
    issue_number = issue["issue_number"]

    # Structured thread text with speaker and timestamp markers, in
    # chronological order. The raw comments file remains the authoritative
    # source for comment-level metadata.
    author = issue.get("issue_author", "") or "unknown"
    association = issue.get("issue_author_association", "") or "NONE"
    created = issue.get("created_at", "") or ""

    parts = [
        f"[ISSUE | {author} ({association}) | {created}]",
        f"TITLE: {issue.get('title', '')}",
        "",
        "BODY:",
        str(issue.get("body", "") or "")
    ]

    if len(comments_df) > 0:
        issue_comments = comments_df[
            (comments_df["repo"] == repo) &
            (comments_df["issue_number"] == issue_number)
        ].sort_values("comment_created_at")

        for j, (_, comment) in enumerate(issue_comments.iterrows(), start=1):
            c_author = comment.get("comment_author", "") or "unknown"
            c_association = comment.get("comment_author_association", "") or "NONE"
            c_created = comment.get("comment_created_at", "") or ""
            c_body = str(comment.get("comment_body", "") or "")

            parts.append("")
            parts.append(f"[COMMENT {j} | {c_author} ({c_association}) | {c_created}]")
            parts.append(c_body)

    full_text = "\n".join(parts).strip()

    matched_terms, matched_dims, matched_levels = attribute_terms(full_text)

    thread_rows.append({
        "repo": repo,
        "repo_category": issue.get("repo_category", ""),
        "issue_number": issue_number,
        "title": issue.get("title", ""),
        "issue_author": issue.get("issue_author", ""),
        "issue_author_association": issue.get("issue_author_association", ""),
        "state": issue.get("state", ""),
        "created_at": issue.get("created_at", ""),
        "updated_at": issue.get("updated_at", ""),
        "closed_at": issue.get("closed_at", ""),
        "comments_count": issue.get("comments_count", 0),
        "labels": issue.get("labels", ""),
        "html_url": issue.get("html_url", ""),
        "zero_comments": issue.get("zero_comments", ""),
        "bundle_dimension": issue.get("bundle_dimension", ""),
        "bundle_level": issue.get("bundle_level", ""),
        "all_bundle_dimensions": issue.get("all_bundle_dimensions", ""),
        "all_bundle_levels": issue.get("all_bundle_levels", ""),
        "n_bundles": issue.get("n_bundles", ""),
        "matched_terms_local": "; ".join(matched_terms),
        "matched_dimensions_local": "; ".join(matched_dims),
        "matched_levels_local": "; ".join(matched_levels),
        "n_matched_terms": len(matched_terms),
        "full_text": full_text
    })

threads_df = pd.DataFrame(thread_rows)

threads_df.to_csv(THREADS_CSV, index=False, encoding="utf-8-sig")

no_local = int((threads_df["n_matched_terms"] == 0).sum())
print(f"Prepared {len(threads_df)} issue threads.")
print(f"Threads with no local term match (tokenisation differences or deleted "
      f"comments): {no_local}")
threads_df.head()


In [ ]:
threads_df_clean = threads_df.applymap(clean_excel_text)
comments_df_clean = comments_df.applymap(clean_excel_text)

threads_df_clean.to_excel(THREADS_EXCEL, index=False)
comments_df_clean.to_excel(COMMENTS_EXCEL, index=False)

print("Saved cleaned Excel files.")


In [ ]:
# Stage-2 sampling frame: OPTIONAL balanced coding sample
#
# This cell never reduces the primary datasets: the candidate, comment, and
# thread files above are always complete. It only adds selection flags for the
# optional Stage-2 coding sample. With MAX_PER_REPO = None (the default), no
# repository cap is applied: every thread is selected and the reserve pool is
# empty. With an integer cap, a reproducible, seeded, stratified selection is
# produced:
#   1. every thread from repositories at or under MAX_PER_REPO is selected;
#   2. repositories over the cap are downsampled to MAX_PER_REPO, stratified
#      proportionally by locally-attributed primary dimension (seed RANDOM_SEED);
#   3. any dimension with fewer than MIN_PER_DIMENSION selected threads
#      (multi-label membership via matched_dimensions_local, falling back to
#      bundle dimensions where no local match exists) is topped up from
#      unselected threads where available;
#   4. unselected threads are flagged as the reserve pool for sensitivity
#      analysis; no rows are dropped.
# The cell also reports each repository's share of the selected set against
# MAX_REPO_SHARE as an advisory check.

import numpy as np

frame = threads_df.copy()
frame["selected_for_coding"] = False

frame["dimension_membership"] = frame["matched_dimensions_local"].fillna("")
fallback = frame["dimension_membership"].str.len() == 0
frame.loc[fallback, "dimension_membership"] = frame.loc[fallback, "all_bundle_dimensions"].fillna("")

strat = frame["matched_dimensions_local"].fillna("").str.split(";").str[0].str.strip()
strat = strat.where(strat.str.len() > 0, frame["bundle_dimension"].fillna("unknown"))
frame["_stratum"] = strat

repo_counts = frame["repo"].value_counts()

if MAX_PER_REPO is None:
    # No repository cap: the full thread corpus is the coding sample.
    frame["selected_for_coding"] = True
    over_cap_repos = []
else:
    # Step 1: repositories at or under the cap
    under_cap = repo_counts[repo_counts <= MAX_PER_REPO].index
    frame.loc[frame["repo"].isin(under_cap), "selected_for_coding"] = True
    over_cap_repos = list(repo_counts[repo_counts > MAX_PER_REPO].index)

# Step 2: proportional stratified downsampling for over-cap repositories
for repo in over_cap_repos:
    sub = frame[frame["repo"] == repo]
    selected_idx = []

    for dim, group in sub.groupby("_stratum"):
        quota = int(round(len(group) / len(sub) * MAX_PER_REPO))
        quota = min(max(quota, 1), len(group))
        selected_idx.extend(
            group.sample(n=quota, random_state=RANDOM_SEED).index.tolist()
        )

    if len(selected_idx) > MAX_PER_REPO:
        rng = np.random.default_rng(RANDOM_SEED)
        selected_idx = list(rng.choice(selected_idx, size=MAX_PER_REPO, replace=False))
    elif len(selected_idx) < MAX_PER_REPO:
        remaining = sub.index.difference(selected_idx)
        top_up = sub.loc[remaining].sample(
            n=min(MAX_PER_REPO - len(selected_idx), len(remaining)),
            random_state=RANDOM_SEED
        ).index.tolist()
        selected_idx.extend(top_up)

    frame.loc[selected_idx, "selected_for_coding"] = True

# Step 3: dimension floors (multi-label membership)
for dim in SYCOPHANCY_TERMS.keys():
    member = frame["dimension_membership"].str.contains(dim, regex=False)
    n_selected = int((member & frame["selected_for_coding"]).sum())

    if n_selected < MIN_PER_DIMENSION:
        pool = frame[member & ~frame["selected_for_coding"]]
        n_needed = min(MIN_PER_DIMENSION - n_selected, len(pool))
        if n_needed > 0:
            top_up = pool.sample(n=n_needed, random_state=RANDOM_SEED)
            frame.loc[top_up.index, "selected_for_coding"] = True

# Step 4: reserve pool flag
frame["reserve_pool"] = ~frame["selected_for_coding"]
frame = frame.drop(columns=["_stratum"])

frame_clean = frame.applymap(clean_excel_text)
frame_clean.to_csv(SAMPLING_FRAME_CSV, index=False, encoding="utf-8-sig")
frame_clean.to_excel(SAMPLING_FRAME_EXCEL, index=False)

n_sel = int(frame["selected_for_coding"].sum())
print(f"Sampling frame saved: {n_sel} of {len(frame)} threads selected; "
      f"{int(frame['reserve_pool'].sum())} in the reserve pool.")

print(f"\nRepository share of the selected set (advisory cap {MAX_REPO_SHARE:.0%}):")
shares = frame[frame["selected_for_coding"]]["repo"].value_counts(normalize=True)
for repo, share in shares.head(10).items():
    flag = "  <-- exceeds advisory share cap" if share > MAX_REPO_SHARE else ""
    print(f"  {repo}: {share:.1%}{flag}")

print("\nSelected threads by dimension (multi-label membership):")
for dim in SYCOPHANCY_TERMS.keys():
    member = frame["dimension_membership"].str.contains(dim, regex=False)
    print(f"  {dim}: {int((member & frame['selected_for_coding']).sum())}")


In [ ]:
files.download(COMMENTS_EXCEL)
files.download(THREADS_EXCEL)
files.download(SAMPLING_FRAME_EXCEL)
files.download(SAMPLING_FRAME_CSV)


In [ ]:
# Final check: collection and sampling funnel for reporting
threads_check = pd.read_excel(THREADS_EXCEL)
frame_check = pd.read_csv(SAMPLING_FRAME_CSV)
log_check = pd.read_csv(SEARCH_LOG_CSV)

raw_hits = log_check.loc[log_check["action"] == "collected", "retrieved"].sum()

print("Reporting funnel")
print(f"  1. Raw bundle hits (before deduplication): {raw_hits}")
print(f"  2. Unique candidate issues: {len(threads_check)}")
print(f"  3. Zero-comment issues (labelled): {int(threads_check['zero_comments'].sum())}")
print(f"  4. Threads with at least one locally attributed term: "
      f"{int((threads_check['n_matched_terms'] > 0).sum())}")
print(f"  5. Selected for screening/coding: {int(frame_check['selected_for_coding'].sum())}")
print(f"  6. Reserve pool for sensitivity analysis: {int(frame_check['reserve_pool'].sum())}")
print("\nSubsequent stages (outside this notebook): false-positive screening,")
print("multi-label coding, and validated sycophancy case counts.")

print("\nThreads by repository category:")
display(threads_check["repo_category"].value_counts())

print("\nThreads by repository (top 15):")
display(threads_check["repo"].value_counts().head(15))

print("\nMost frequently locally-attributed terms (top 20):")
term_series = (
    threads_check["matched_terms_local"].fillna("")
    .str.split("; ").explode()
)
display(term_series[term_series.str.len() > 0].value_counts().head(20))


In [ ]:
print("Final reproducibility settings")
print(f"Repositories: {len(REPOS)}")
n_terms = sum(len(tl) for d in SYCOPHANCY_TERMS.values() for tl in d.values())
print(f"Search terms: {n_terms} in bundled OR-queries (core / exploratory layers)")
print("Search scope: issue titles, bodies, and comments (in:title,body,comments)")
print(f"Eligibility window (issue creation date): {START_DATE.isoformat()} to {END_DATE.isoformat()}")
print(f"Search sleep seconds: {SEARCH_SLEEP_SECONDS}")
print(f"Result cap handling: recursive date-window splitting at {API_RESULT_CAP}; "
      f"near-cap flag at {NEAR_CAP_THRESHOLD}")
print(f"Sampling frame: max {MAX_PER_REPO} per repository, minimum {MIN_PER_DIMENSION} "
      f"per dimension, advisory repo share cap {MAX_REPO_SHARE:.0%}, seed {RANDOM_SEED}")
